# 15 — The Alberta director package built from the Y2Y-WIDE analysis within the Alberta extent (the same package as 12, from the parent's runs; the outputs are `16`)

**What this is.** The SAME director package as `12_tiers_and_clusters` (package spec v2.1: the core = the balanced scenario's guarded tier; the
same tables, geotiffs, clusters, picks), built from the **Y2Y-wide
guarded sweeps** (`analyses/y2y/runs_v4/`, the flagship's 14 design formulations × 51 plans, `manifest_v4.csv`) **read on the Alberta PU**
instead of from the Alberta run (`runs_v4/ab_l/A/`). Both analyses share one 1 km grid with identical cell IDs (M2.2), so every Y2Y-wide
plan restricts to Alberta by masking, no resampling. Zero solves. Outputs → `director_package_y2y_analysis/` (geotiffs, tables,
summary.json); the presentation set is **`16_director_outputs_y2y_analysis`** (the exact files of 14, rendered from this package without re-running
anything here). Kernel `y2y-geo`; ~2 min.

**What "the Y2Y-wide analysis within Alberta" means here (disclosed, every choice is one line):**

| | Alberta run (12 → 14) | this notebook |
|---|---|---|
| plans | the Alberta MILP: level-A budget (locked + 10,083 km²), Alberta-derived weights, Alberta targets (D-AB5/D-AB12/D-AB14) | the Y2Y-wide MILP: 30% of the whole Y2Y region, Y2Y-wide weights and targets — Alberta gets whatever the corridor-wide optimum places in it (an OUTCOME, not a budget) |
| members / band | guarded 5% at the applied band (D-AB13) | the parent's guarded 5% (the estimand, parent M4.38); no flatness fallback — D-AB13 is an Alberta-run rule |
| unguarded band | reference cell only | the same (the parent's reference cell `s0_ssp585_theta5` at 2 / 5 / 10%) |
| core / tiers / clusters | balanced f ≥ 0.70; 10 km² clusters, 10 km complexes, 20 km regional picks (D-AB7) | the same thresholds and display rules on the clipped surfaces, so the two decks are comparable side by side |
| star axes / consequences / value maps | percentiles and "×" ratios over ALBERTA's allocatable land | the same Alberta denominators (a cluster is scored against the land the Alberta program can act on, whichever analysis found it); the parent's own deck scores Y2Y-wide |
| representativeness | 13 classes present in Alberta; rare-in-window on the Alberta + 250 km window (D-AB12) | the same Alberta rule for the value vote; the parent's 20-class block and Y2Y + 250 km targets are what the plans were solved under (the objectives table says so) |
| E11, E19, E17 | Alberta records (11, 11b/11c) | the parent's records: E11 is a property of the formulations Y2Y-wide (carried as is); E19 forced cells and E17 arms read on the Alberta PU |
| captures (T-D3 / T-D5) | Alberta anchors' share of Alberta totals | the Y2Y-wide anchors' share of ALBERTA totals inside Alberta (written as `T1_*_within_alberta.csv`; the parent's Y2Y-wide captures are in its own record) |
| comparison | the v3.1 picks scored against v4 (R9.6) | **this package against the Alberta run's** (`tables/comparison_*.csv`, R9.8): core areas, overlap, each deck cluster's share inside the other analysis's core, the tier cross-tab |

Cells 1–9 = 12's cells, source-swapped. Nothing here touches `director_package/` (the Alberta run's package) or the parent's.


In [1]:
import importlib, json, pathlib, shutil, sys
from types import SimpleNamespace
import numpy as np
import pandas as pd
import rasterio
import geopandas as gpd
from rasterio import features as rfeatures
from scipy import ndimage

_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.py").exists()]
assert _cands, "config.py not found above the notebook"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))
import config, leverage_core as lc, ensemble_core as ec, director_core as dc
for _m in (config, lc, ec, dc):
    importlib.reload(_m)

HERE = ROOT / "analyses" / "alberta_prioritization"
VP = config.ab_paths(); VERSION = VP.version
assert config.EFG_SUBDIR == VP.efg_subdir, f"config.EFG_SUBDIR {config.EFG_SUBDIR!r} does not match VERSION {VERSION}"
SPEC, DATA, AB4, REC = HERE / "spec", HERE / "data", VP.analysis, VP.records
PKG = HERE / "director_package_y2y_analysis"                    # THIS package: the Y2Y-wide analysis within Alberta (never director_package/, the Alberta run's)
PKG_AB = HERE / "director_package"                             # the Alberta run's package (12): the comparison partner in cell 9
assert VERSION == "v4", f"the Y2Y-wide-within-Alberta package is defined on manifest v4 (the registered design), config is {VERSION}"
# supersede, never delete: a package built on another manifest version is archived before this one is written
if (PKG / "summary.json").exists():
    _oldS = json.loads((PKG / "summary.json").read_text()); _old = _oldS.get("version", "v1")     # the 2026-09-08 build carried no version key
    _old_basis = _oldS.get("core_basis", "ensemble")                                   # package spec v2.1: the core basis is part of the package identity
    if _old != VERSION or _old_basis != "balanced":
        _arch = PKG / (f"_superseded_{_old}_artifact_block" if _old == "v1" else (f"_superseded_{_old}" if _old != VERSION else f"_superseded_{_old}_{_old_basis}_core")); _arch.mkdir(exist_ok=True)
        for _item in list(PKG.iterdir()):
            if not _item.name.startswith("_"):
                shutil.move(str(_item), str(_arch / _item.name))
        print(f"archived the {_old} package to {_arch.relative_to(ROOT)}")
for sub in ("geotiffs", "tables", "figures"):
    (PKG / sub).mkdir(parents=True, exist_ok=True)
GEO, TAB = PKG / "geotiffs", PKG / "tables"
AB = VP.stack                                            # the version's Alberta stack (aligned_stack_ab_v4/ under v4)
RUNS = dc.RUNS                                           # THE SOURCE SWAP: the Y2Y-wide runs (analyses/y2y/runs_v4/), read on the Alberta PU below
assert RUNS.name == f"runs_{VERSION}" and RUNS.parent.name == "y2y", RUNS
EXTENT = json.loads((SPEC / "ab_extent_v1.json").read_text())
SC = json.loads((dc.SPEC / "scenarios_v4.json").read_text())          # the Y2Y-wide scenarios (weights / targets the plans were solved under)
MAN_ALL = pd.read_csv(dc.MANIFEST); assert len(MAN_ALL) == 14, len(MAN_ALL)   # the Y2Y-wide manifest v4 (14 design rows)
BLOCKS = json.loads(MAN_ALL.block_structure.iloc[0]) if "block_structure" in MAN_ALL.columns else SC["_meta"]["blocks"]   # five under v4 (the manifest is the record)
assert list(BLOCKS) == list(config.BLOCKS), (list(BLOCKS), list(config.BLOCKS))
MAN = dc.package_manifest(MAN_ALL)                     # the design positions (12 up to v3.1, 14 under v4; `role` column)
FORMS = list(MAN.formulation_id)
N_EFG_Y2Y = len(lc.efg_paths(config.Y2Y_STACK_DIR))                   # the block the plans were solved under (20 classes Y2Y-wide; 13 present in Alberta)
print(f"VERSION {VERSION}: Y2Y-wide {dc.MANIFEST.name}; runs {RUNS.relative_to(ROOT)} read on the Alberta PU of {AB.relative_to(ROOT)}; "
      f"EFG block {config.EFG_SUBDIR} ({N_EFG_Y2Y} features Y2Y-wide, {len(lc.efg_paths(AB))} in Alberta)")

# ---- AB constants (D-AB7, disclosed in M13) --------------------------------------------------------------
BAND = "g05"                                          # the parent's guarded 5% band = the estimand (parent M4.38); D-AB13 is a rule of the Alberta RUN, not applied here
MIN_KM2, COMPLEX_LINK_KM, PICK_LINK_KM = 10, 10, 20   # parent 100 km2 / 25 km / 50 km (deck picks: regional clusters, 2x the complex link since the parent's 75 -> 50 km, M4.40 addendum; here 30 -> 20 km, D-AB7)
SPECK_LINK_KM = dc.SPECK_LINK_KM                     # 10 km, as the parent (M4.33 k): sub-floor core components within it join the nearest regional cluster
# the deck's core-cluster count (parent M4.40 addendum: dc.MAX_CORE_CLUSTERS = 4, positional in the north -> south numbering -- Ethan's "remove cluster 5" for
# Greater Yellowstone). On Alberta the balanced core carries FIVE regional clusters and the positional rule would drop the southernmost, SE of Waterton
# (147 km2, the Crown of the Continent), while keeping a 66 km2 piece south of Don Getty -- so the Alberta default shows all five (None); set 4 for the literal mirror.
MAX_CORE_CLUSTERS_AB = None
# deck picks merged by hand (Ethan 2026-09-30, on the 15 package's north -> south numbering: the pairs sit 18 km and 11 km apart, but a link distance that
# joins them (>= 33 km) would chain 2-3-4-5 into one) -- pairs of regional-cluster numbers merged after the regional grouping, then renumbered north -> south;
# [] = none. Merged rows take the larger member's name and anchor; area, mean f, lat/lon combine area-weighted as in dc.group_picks.
MERGE_CORE_PICKS = [(2, 3), (4, 5)]
# per-cluster region words (the communities lookup names a whole census division; Division No. 15 runs 49-54 degN, so a cluster inside it takes its own
# word here -- the lookup's own VET note). Keyed by (act, FINAL number after the merges); Ethan supplies the words; {} = the lookup word.
CLUSTER_REGION_OVERRIDES = {("Act 1", "1"): "Upper Smoky",
                            ("Act 1", "2"): "Alberta Rockies",                # 93% Rocky Mountain natural region (subalpine / alpine), 7% foothills: the front ranges east of the parks, not the foothills the census word says
                            ("Act 1", "3"): "Kananaskis / Highwood"}         # PROPOSED 2026-09-30 for the merged Don Getty pair; lookup word = 'Alberta Rockies'
THR = dc.FREQ_THR
FLOOR_G = dc.FLOOR_G
N_EFG = len(lc.efg_paths(AB))

# ---- the AB grid as a director_core-compatible object -----------------------------------------------------
with rasterio.open(AB / "cost_uniform.tif") as src:
    tr, shp, prof = src.transform, src.shape, src.profile
pu = lc.pu_mask(AB)
with rasterio.open(AB / "mask_protected_areas.tif") as src:
    locked2d = (src.read(1) == 1) & pu
G = SimpleNamespace(pu=pu, locked2d=locked2d, locked=locked2d[pu], disc=~locked2d[pu], n_pu=int(pu.sum()),
                    n_disc=int((~locked2d[pu]).sum()), shape=shp, transform=tr, crs=config.TARGET_CRS, profile=prof,
                    cell_km2=abs(tr.a * tr.e) / 1e6)
G.rows, G.cols = np.where(pu)
assert G.n_pu == EXTENT["n_pu"]
# no Alberta budget level here: the Y2Y-wide plans select 30% of the whole region and place what they place in Alberta (measured after the load)
SUMMARY = dict(version=VERSION, manifest=dc.MANIFEST.name, manifest_source=str(dc.MANIFEST.relative_to(ROOT)), runs_source=str(RUNS.relative_to(ROOT)),
               package_basis="Y2Y-wide analysis (the flagship's 14 guarded sweeps) read within the Alberta extent; Alberta denominators for percentiles and ratios",
               efg_block=config.EFG_SUBDIR, n_efg=N_EFG, n_efg_y2y=N_EFG_Y2Y, pick_link_km=PICK_LINK_KM, speck_link_km=SPECK_LINK_KM,
               core_basis="balanced",                                                     # package spec v2.1 (parent M4.40; AB v0.6.1)
               core_definition="guarded f of the balanced scenario (S0) averaged over SSP245 and SSP585, >= 0.70; ensemble F = the paper's estimand, appendix",
               max_core_clusters=MAX_CORE_CLUSTERS_AB,
               efg_target_rule=(str(MAN_ALL.efg_target_rule.iloc[0]) if "efg_target_rule" in MAN_ALL.columns else "flat"),
               applied_band_g=0.05, applied_band_rule="the parent's guarded 5% band (the estimand, M4.38); no Alberta flatness fallback (D-AB13 not applied)",
               n_formulations=len(FORMS), threshold=THR, floor_g=FLOOR_G, min_km2=MIN_KM2, complex_link_km=COMPLEX_LINK_KM,
               band=BAND, level="Y2Y-wide (30% of the whole region; no Alberta level)", budget_pct_y2y=float(MAN_ALL.budget_pct.iloc[0]),
               pa_km2=int(G.locked.sum()), pa_pct_of_extent=float(100 * G.locked.sum() / G.n_pu))

# ---- load every Y2Y-wide formulation once ON THE ALBERTA PU (the parent's own reader, director_core.load_guarded, over G = the Alberta grid:
#      f, unions and D are computed on the Alberta cells of each Y2Y-wide plan; certificates and k = 50 asserted; the unguarded band on the reference cell only) ----
L = dc.load_guarded(G, MAN)                                  # raises if any of the 14 guarded sweeps is missing
assert L.forms == FORMS and not L.missing
L.plain_forms = [f for f in FORMS if f in L.f_plain]
PLAIN_ALL = set(L.plain_forms) == set(FORMS); REF = "s0_ssp585_theta5"
cert = pd.DataFrame(L.cert).T; cert["D_plain"] = pd.Series(L.D_plain); cert["D_guard"] = pd.Series(L.D_guard)
# what the Y2Y-wide plans place in Alberta (an OUTCOME of the corridor-wide optimum, not a budget): additions = anchor cells on unprotected Alberta land
ADD_BY_FORM = {fid: int((L.anchors[fid] & G.disc).sum()) for fid in FORMS}
ADD_KM2 = ADD_BY_FORM[REF]                                   # the reference cell (balanced, SSP5-8.5) = the headline number
BUDGET_PCT_A = float(L.anchors[REF].sum() / G.n_pu)          # the share of the Alberta extent the reference plan selects (locked + additions); T-D5's "new half" denominator
SUMMARY.update(additions_km2=ADD_KM2, additions_km2_by_formulation=ADD_BY_FORM, additions_km2_range=[min(ADD_BY_FORM.values()), max(ADD_BY_FORM.values())],
               budget_pct=BUDGET_PCT_A, additions_basis=f"cells of the Y2Y-wide anchor on unprotected Alberta land ({REF} = the headline; range over the 14 positions)",
               alberta_run_additions_km2=EXTENT["additions_cells"])
print(f"package: {len(FORMS)} Y2Y-wide positions at g={BAND[1:]}% (guarded = deliverable) read within Alberta | PU {G.n_pu:,} | unprotected {G.n_disc:,} | "
      f"PAs {G.locked.sum():,} km2 = {SUMMARY['pa_pct_of_extent']:.1f}% of the extent")
print(f"Alberta additions of the Y2Y-wide plans (anchor cells on unprotected Alberta land): {REF} {ADD_KM2:,} km2; range {min(ADD_BY_FORM.values()):,}-{max(ADD_BY_FORM.values()):,} km2 "
      f"over the 14 positions | the Alberta run's level-A budget: {EXTENT['additions_cells']:,} km2")
print(cert.to_string(float_format=lambda v: f"{v:.3f}"))


VERSION v4: Y2Y-wide manifest_v4.csv; runs analyses/y2y/runs_v4 read on the Alberta PU of input_data/aligned_stack_ab_v4; EFG block iucn_efg_v3 (20 features Y2Y-wide, 13 in Alberta)
s0_ssp585_theta5       f_guard freq   2,649 km2 | plain      77 km2 | D 1.293 -> 1.008
s1_ssp585_theta5       f_guard freq   4,097 km2 | plain      (none) | D_guard 0.860
s2_ssp585_theta5       f_guard freq   3,349 km2 | plain      (none) | D_guard 0.705
s2c_ssp585_theta5      f_guard freq   1,050 km2 | plain      (none) | D_guard 1.650
s3_ssp585_theta5       f_guard freq   1,650 km2 | plain      (none) | D_guard 1.393
s4_ssp585_theta3       f_guard freq   1,999 km2 | plain      (none) | D_guard 1.171
s5_ssp585_theta5       f_guard freq   2,237 km2 | plain      (none) | D_guard 1.144
s0_ssp245_theta5       f_guard freq   2,030 km2 | plain      (none) | D_guard 0.938
s1_ssp245_theta5       f_guard freq   1,924 km2 | plain      (none) | D_guard 0.877
s2_ssp245_theta5       f_guard freq   3,014 km2 | plain    

In [2]:
# ---- surfaces: THE CORE = the balanced scenario's guarded f (spec v2.1); ensemble F beside it (the paper); unguarded reference; by refugia future ----
Fg = dc.ensemble(L.f_guard, FORMS); Ug = dc.union_membership(L, FORMS, True)              # ensemble F over the design cells = the paper's Claim-A estimand
dc.write_tif(G, Fg, GEO / "F_guarded.tif"); dc.write_tif(G, Ug, GEO / "union_membership_guarded.tif")
# the UNGUARDED band: every formulation up to v3.1 (an ensemble F_unguarded beside the guarded one); under manifest v4 the
# reference cell only -- then the side-by-side is that cell's own f, labelled as such, never an ensemble (parent 19 mirror)
if PLAIN_ALL:
    Fp = dc.ensemble(L.f_plain, FORMS); Up = dc.union_membership(L, FORMS, False); PLAIN_LABEL = "unguarded"
    dc.write_tif(G, Fp, GEO / "F_unguarded.tif")
elif REF in L.f_plain:
    Fp = L.f_plain[REF]; Up = L.union_plain[REF].astype(np.float32); PLAIN_LABEL = f"unguarded ({REF} only)"
    dc.write_tif(G, Fp, GEO / "f_unguarded_reference.tif")
else:
    Fp = Up = None; PLAIN_LABEL = "unguarded (none)"
SUMMARY["unguarded_basis"] = "all formulations" if PLAIN_ALL else PLAIN_LABEL
for fid in FORMS:
    dc.write_tif(G, L.f_guard[fid], GEO / f"f_guarded_{fid}.tif")
# THE CORE SURFACE (package spec v2.1 / study plan v0.20.1, parent M4.40): the BALANCED scenario's guarded f, its two climate levels averaged (one vote each)
SID_OF = MAN.set_index("formulation_id").scenario_id
S0_FORMS = [f for f in FORMS if SID_OF[f] == "s0"]
assert len(S0_FORMS) == 2, f"expected the balanced scenario at both climate levels, got {S0_FORMS}"
F0 = np.mean([L.f_guard[f] for f in S0_FORMS], axis=0).astype(np.float32)
dc.write_tif(G, F0, GEO / "f_balanced_core.tif")
core0 = (F0 >= THR) & G.disc; core_ens = (Fg >= THR) & G.disc
print(f"THE CORE (balanced scenario, both futures averaged, f >= {THR}): {int(core0.sum()):,} km2 | ensemble F core (paper, appendix): "
      f"{int(core_ens.sum()):,} km2 | ensemble core inside the balanced core {100 * (core0 & core_ens).sum() / max(core_ens.sum(), 1):.1f}% | Jaccard {dc.jaccard(core0, core_ens):.3f}")
# by climate future: the balanced scenario at each refugia realization (the SSP-only variants, spec v2.1) feed the by-future products
# downstream (F_LEV); the ensemble F per level is still written for the appendix record
LEVELS = {"585": [f for f in FORMS if "ssp585" in f], "245": [f for f in FORMS if "ssp245" in f]}
F_LEV_ENS = {lv: dc.ensemble(L.f_guard, fids) for lv, fids in LEVELS.items() if fids}
for lv, F in F_LEV_ENS.items():
    dc.write_tif(G, F, GEO / f"F_guarded_ssp{lv}.tif")
F_LEV = {lv: L.f_guard[f] for lv in ("585", "245") for f in S0_FORMS if f"ssp{lv}" in f}
for lv, F in F_LEV.items():
    dc.write_tif(G, F, GEO / f"f_balanced_ssp{lv}.tif")
SUMMARY["by_level"] = {}
for lv, F in F_LEV.items():
    c = (F >= THR) & G.disc
    SUMMARY["by_level"][lv] = dict(n=1, core_km2=int(c.sum()), jaccard_vs_core=dc.jaccard(c, core0),
                                   pct_of_core_inside=float(100 * (c & core0).sum() / max(core0.sum(), 1)))
c5, c2 = [(F_LEV[l] >= THR) & G.disc for l in ("585", "245")]
SUMMARY["by_level"]["jaccard_585_245"] = dc.jaccard(c5, c2); SUMMARY["by_level"]["union_km2"] = int((c5 | c2).sum()); SUMMARY["by_level"]["intersection_km2"] = int((c5 & c2).sum())
SUMMARY["by_level_ensemble"] = {lv: dict(n=len(LEVELS[lv]), core_km2=int(((F >= THR) & G.disc).sum()),
                                         pct_of_ensemble_core_inside=float(100 * (((F >= THR) & G.disc) & core_ens).sum() / max(core_ens.sum(), 1)))
                                for lv, F in F_LEV_ENS.items()}
print(f"\nthe balanced scenario by climate future vs its averaged core ({int(core0.sum()):,} km2):\n" + json.dumps(SUMMARY["by_level"], indent=1, default=float))
TD2a = dc.band_table(G, {"balanced core (applied)": F0, "ensemble F (paper)": Fg, **({PLAIN_LABEL: Fp} if Fp is not None else {}),
                         **{f"balanced - SSP{lv} only": F for lv, F in F_LEV.items()}})
TD2a.to_csv(TAB / "T-D2_bands.csv", index=False)
print("\nT-D2 (bands, unprotected land):"); print(TD2a.to_string(index=False, float_format=lambda v: f"{v:.1f}"))
SUMMARY["frequent_km2"] = dict(guarded=int(core0.sum()), ensemble=int(core_ens.sum()), unguarded=(int(((Fp >= THR) & G.disc).sum()) if Fp is not None else None))
SUMMARY["always_km2"] = dict(guarded=int(((F0 >= 0.95) & G.disc).sum()), ensemble=int(((Fg >= 0.95) & G.disc).sum()), unguarded=(int(((Fp >= 0.95) & G.disc).sum()) if Fp is not None else None))
SUMMARY["ensemble_core_inside_balanced_pct"] = float(100 * (core0 & core_ens).sum() / max(core_ens.sum(), 1))
SUMMARY["ever_in_band_pct"] = dict(guarded=float(100 * (Ug[G.disc] > 0).mean()), unguarded=(float(100 * (Up[G.disc] > 0).mean()) if Up is not None else None))
p10 = RUNS / "s0_ssp585_theta5" / "mga_g10.tif"
if p10.exists():
    SUMMARY["s0_g10_union_pct"] = float(100 * ec.read_selections(p10, G.pu).any(axis=0)[G.disc].mean())
# 5%-band context for the guardrail sentence (R5/R7: every unprotected cell is in some 5% plan): the plain 5% band where every
# formulation has one (v3.1 / v1), else the guarded 5% band (v4: the guarded band is the estimand; plain exists on the reference cell only)
u5_tag = "mga_g05.tif" if all((RUNS / f / "mga_g05.tif").exists() for f in FORMS) else "mga_guard_g05.tif"
u5 = np.mean([ec.read_selections(RUNS / f / u5_tag, G.pu).any(axis=0) for f in FORMS], axis=0)
SUMMARY["ever_in_5pct_band_pct"] = float(100 * (u5[G.disc] > 0).mean()); SUMMARY["ever_in_5pct_band_basis"] = "plain" if u5_tag == "mga_g05.tif" else "guarded"
D11 = pd.read_csv(dc.SPEC_REC / "E11_delta_matrix.csv", index_col=0).loc[FORMS, FORMS]   # E11 is a property of the formulations' objectives Y2Y-wide (the parent's record, carried as is)
off = ~np.eye(len(D11), dtype=bool)
SUMMARY["e11_pairs_in_band"] = [int((D11.values[off] <= 0.05 + 1e-9).sum()), int(off.sum())]
print(f"\never in a {BAND[1:]}% guarded band: {SUMMARY['ever_in_band_pct']['guarded']:.1f}% of unprotected land | in a 5% {SUMMARY['ever_in_5pct_band_basis']} band: {SUMMARY['ever_in_5pct_band_pct']:.1f}% | "
      f"E11 {SUMMARY['e11_pairs_in_band'][0]}/{SUMMARY['e11_pairs_in_band'][1]} pairs mutually near-optimal")


THE CORE (balanced scenario, both futures averaged, f >= 0.7): 2,141 km2 | ensemble F core (paper, appendix): 1,756 km2 | ensemble core inside the balanced core 100.0% | Jaccard 0.820

the balanced scenario by climate future vs its averaged core (2,141 km2):
{
 "585": {
  "n": 1,
  "core_km2": 2649,
  "jaccard_vs_core": 0.7839851024208566,
  "pct_of_core_inside": 98.31854273703877
 },
 "245": {
  "n": 1,
  "core_km2": 2030,
  "jaccard_vs_core": 0.7748936170212766,
  "pct_of_core_inside": 85.05371321812237
 },
 "jaccard_585_245": 0.6167933655839668,
 "union_km2": 2894,
 "intersection_km2": 1785
}

T-D2 (bands, unprotected land):
                    band  balanced core (applied) km2  balanced core (applied) %disc  ensemble F (paper) km2  ensemble F (paper) %disc  unguarded (s0_ssp585_theta5 only) km2  unguarded (s0_ssp585_theta5 only) %disc  balanced - SSP585 only km2  balanced - SSP585 only %disc  balanced - SSP245 only km2  balanced - SSP245 only %disc
      never [0.00, 0.05)         

In [3]:
# ---- pooling (force: one map per scenario), act tiers, Act-2 ownership, T-D2 acts ----------------------------------
POOL, rep = dc.pool_scenarios(G, L.f_guard, MAN, force=True)
POOLp = dc.pool_scenarios(G, L.f_plain, MAN, force=True)[0] if PLAIN_ALL else {}
rep.to_csv(TAB / "pooling_check.csv", index=False)
print("pooling check (frequent-tier Jaccard between refugia futures; rule pools iff >= 0.80; deck forces one map per scenario):")
print(rep.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
SUMMARY["pooling"] = rep.to_dict(orient="records")

CORE_KEY = "Act 1 core (balanced scenario, f >= 0.70, both climate levels averaged)"          # package spec v2.1
ENS_KEY = f"appendix: ensemble core (F >= 0.70, all {len(FORMS)} design formulations -- the paper's estimand)"
assert np.allclose(POOL["s0"], F0, atol=1e-6), "the pooled balanced f must equal the core surface"
def act_masks(Fx, POOLx, Ux):
    core = (Fx >= THR) & G.disc
    out = {CORE_KEY: core}
    claimed = core.copy(); any_sc = np.zeros(G.n_pu, bool)
    for key, f in POOLx.items():
        sid = key.split("@")[0]
        if sid == "s0":                                    # the balanced scenario IS the core; its "frequent minus core" is empty by construction
            continue
        tier = (f >= THR) & G.disc & ~core
        tag = "Act 2" if sid in dc.ACT2_SCENARIOS else "appendix"
        out[f"{tag} {key}: {dc.SCENARIO_LABEL[sid]} (frequent minus core)"] = tier
        if sid in dc.ACT2_SCENARIOS:
            any_sc |= tier
    out["Act 2 any named scenario (union)"] = any_sc; claimed |= any_sc
    out["Act 3 opportunity (in >= 1 band, not above)"] = (Ux > 0) & G.disc & ~claimed
    out["never (no near-optimal plan selects it)"] = (Ux == 0) & G.disc
    return out
AG = act_masks(F0, POOL, Ug); AP = act_masks(Fp, POOLp, Up) if PLAIN_ALL else {}
AG[ENS_KEY] = (Fg >= THR) & G.disc
for sid in dc.ACT2_SCENARIOS:                              # the balanced-core land each lean gives up (spec v2.1: "reported, since it is now visible")
    if sid in POOL:
        AG[f"Act 2 {sid}: balanced-core land the lean gives up ({dc.SCENARIO_LABEL[sid]}: core, not frequent under it)"] = AG[CORE_KEY] & ~((POOL[sid] >= THR) & G.disc)
AG["Act 1 core - BOTH refugia futures (cell-level intersection)"] = c5 & c2
AG["climate-conditional core - SSP585 future only"] = c5 & ~c2
AG["climate-conditional core - SSP245 future only"] = c2 & ~c5
rows = []
for k in AG:
    kp = AP.get(k)
    rows.append({"tier": k, "guarded km2": int(AG[k].sum()), "guarded %disc": 100 * AG[k].sum() / G.n_disc,
                 "unguarded km2": int(kp.sum()) if kp is not None else np.nan, "unguarded %disc": 100 * kp.sum() / G.n_disc if kp is not None else np.nan})
TD2b = pd.DataFrame(rows)
core_m = AG[CORE_KEY]                                                                  # the balanced core
owner = np.zeros(G.n_pu, np.uint8); nq = np.zeros(G.n_pu, np.uint8); best = np.full(G.n_pu, -1.0, np.float32)
for i, sid in enumerate(dc.ACT2_SCENARIOS, 1):
    if sid not in POOL: continue
    q = (POOL[sid] >= THR) & G.disc & ~core_m; nq[q] += 1
    take = q & (POOL[sid] > best); owner[take] = i; best[take] = POOL[sid][take]
MULTI = dc.MULTI_OWNER; owner[nq >= 2] = MULTI                 # 99 under v4 (a fifth scenario would collide with the old 5)
dc.write_tif(G, owner, GEO / "act2_owner.tif", dtype="uint8", nodata=255)
own_rows = [{"tier": f"Act 2 by scenario - {dc.SCENARIO_LABEL[s]} only", "guarded km2": int((owner == i).sum()), "guarded %disc": 100 * (owner == i).sum() / G.n_disc,
             "unguarded km2": np.nan, "unguarded %disc": np.nan} for i, s in enumerate(dc.ACT2_SCENARIOS, 1)]
own_rows.append({"tier": "Act 2 by scenario - frequent under 2+ named scenarios", "guarded km2": int((owner == MULTI).sum()), "guarded %disc": 100 * (owner == MULTI).sum() / G.n_disc, "unguarded km2": np.nan, "unguarded %disc": np.nan})
TD2b = pd.concat([TD2b, pd.DataFrame(own_rows)], ignore_index=True); TD2b.to_csv(TAB / "T-D2_acts.csv", index=False)
SUMMARY["act2_owner_km2"] = {dc.SCENARIO_LABEL[s]: int((owner == i).sum()) for i, s in enumerate(dc.ACT2_SCENARIOS, 1)} | {"2+ scenarios": int((owner == MULTI).sum())}
print("\nT-D2 (act tiers):"); print(TD2b.to_string(index=False, float_format=lambda v: f"{v:.1f}"))
tiers = np.zeros(G.n_pu, np.uint8)
tiers[AG["Act 3 opportunity (in >= 1 band, not above)"]] = 1; tiers[AG["Act 2 any named scenario (union)"]] = 2; tiers[core_m] = 3
dc.write_tif(G, tiers, GEO / "act_tiers_guarded.tif", dtype="uint8", nodata=255)


pooling check (frequent-tier Jaccard between refugia futures; rule pools iff >= 0.80; deck forces one map per scenario):
scenario  levels  jaccard                                                   decision  freq_km2_585  freq_km2_245
      s0       2    0.617 POOLED (forced; rule would separate at Jaccard 0.62 < 0.8)          2649          2030
      s1       2    0.265 POOLED (forced; rule would separate at Jaccard 0.26 < 0.8)          4097          1924
      s2       2    0.870                                                     POOLED          3349          3014
     s2c       2    0.610 POOLED (forced; rule would separate at Jaccard 0.61 < 0.8)          1050           678
      s3       2    0.572 POOLED (forced; rule would separate at Jaccard 0.57 < 0.8)          1650          1139
      s4       2    0.584 POOLED (forced; rule would separate at Jaccard 0.58 < 0.8)          1999          1319
      s5       2    0.690 POOLED (forced; rule would separate at Jaccard 0.69 < 0.8)    

PosixPath('/Users/ethanberman/Dropbox/Python Projects/y2y-spatial-optimization/analyses/alberta_prioritization/director_package_y2y_analysis/geotiffs/act_tiers_guarded.tif')

In [4]:
# ---- named context: PAs in the extent + the two AOIs (the IPCA analogue); clustering + complexes + picks ----------------
ext = gpd.read_file(DATA / "ab_extent_v1.gpkg").to_crs(G.crs)
pa = gpd.clip(gpd.read_file(config.PA_VECTOR).to_crs(G.crs), ext).dissolve(by="PA_Name").reset_index(); pa["km2"] = pa.area / 1e6
zones = gpd.read_file("/vsizip/" + str(DATA / "aoi" / "US_SRP_Zones.zip") + "/Data/US_SRP_Zones.shp").to_crs(G.crs)
srp = gpd.read_file("/vsizip/" + str(DATA / "aoi" / "US_SRP_PlanningBoundary.zip") + "/Data/US_SRP_PlanningBoundary.shp").to_crs(G.crs)
AOI = gpd.GeoDataFrame({"name": ["Upper Smoky Nature-First zone", "Upper Smoky SRP planning area"],
                        "geometry": [zones[zones.Zone == "Nature First"].union_all(), srp.union_all()]}, crs=G.crs)
AOI.to_file(GEO / "aoi.gpkg", driver="GPKG")
AOI_MASK = {n: rfeatures.rasterize([(g, 1)], out_shape=G.shape, transform=G.transform, fill=0, dtype="uint8").astype(bool) & G.pu
            for n, g in zip(AOI.name, AOI.geometry)}
named = gpd.GeoDataFrame(pd.concat([pa[pa.km2 >= 25][["PA_Name", "geometry"]].rename(columns={"PA_Name": "name"}), AOI], ignore_index=True), geometry="geometry", crs=G.crs)
core2d = dc.to_grid(G, core_m, fill=False, dtype=bool)

lab1, reg1 = dc.clusters(G, F0, min_km2=MIN_KM2)                                   # the balanced core (spec v2.1)
reg1.insert(0, "act", "Act 1"); reg1.insert(1, "key", "core")
reg1["name"] = [dc.placeholder_name(G, lab1 == c, named) if k else "" for c, k in zip(reg1.cid, reg1.kept)]
sens = [dc.sensitivity(G, F0, min_km2=MIN_KM2).assign(act="Act 1", key="core")]
print(f"Act 1 (balanced core): tier {int(core_m.sum()):,} km2 -> {len(reg1)} components, {int(reg1.kept.sum())} >= {MIN_KM2} km2 ({reg1[reg1.kept].km2.sum():,.0f} km2)")
LABELS = {"act1": lab1}
reg1, cx1 = dc.group_complexes(G, lab1, reg1, link_km=COMPLEX_LINK_KM)
REGS, CX, PICKS, number = [reg1], {"core": cx1}, [], 0
def pick_row(number, act, key, c, names):
    return dict(number=number, act=act, key=key, cid=int(c.anchor_cid), cids=";".join(map(str, c.cids)), n_components=int(c.n),
                name=names.get(int(c.anchor_cid), ""), km2=float(c.km2), meanF=float(c.meanF), lat=float(c.lat), lon=float(c.lon))
names1 = dict(zip(reg1.cid.astype(int), reg1["name"]))
for _, c in cx1.head(dc.TOPK_ACT1).iterrows():
    number += 1; PICKS.append(pick_row(number, "Act 1", "core", c, names1))
print(f"Act 1 complexes (single linkage {COMPLEX_LINK_KM} km): {len(cx1)} from {int(reg1.kept.sum())} components; top {dc.TOPK_ACT1}: "
      + ", ".join(f"{int(c.km2):,} km2 ({int(c.n)} comp.)" for _, c in cx1.head(dc.TOPK_ACT1).iterrows()))
for key, f in POOL.items():
    sid = key.split("@")[0]
    if sid not in dc.ACT2_SCENARIOS: continue
    lab, reg = dc.clusters(G, f, min_km2=MIN_KM2, subtract2d=core2d)
    reg.insert(0, "act", "Act 2"); reg.insert(1, "key", key)
    reg["name"] = [dc.placeholder_name(G, (lab == c) & ~core2d, named) if k else "" for c, k in zip(reg.cid, reg.kept)]
    reg, cxk = dc.group_complexes(G, lab, reg, link_km=COMPLEX_LINK_KM)
    LABELS[f"act2_{key}"] = lab; REGS.append(reg); CX[key] = cxk
    sens.append(dc.sensitivity(G, f, min_km2=MIN_KM2, subtract2d=core2d).assign(act="Act 2", key=key))
    kept = reg[reg.kept]
    print(f"Act 2 {key:<4} ({dc.SCENARIO_LABEL[sid]}): {len(reg)} components, {len(kept)} kept after core subtraction "
          f"({kept.residual_km2.sum() if len(kept) else 0:,.0f} km2 residual) -> {len(cxk)} complexes")
    namesk = dict(zip(reg.cid.astype(int), reg["name"]))
    for _, c in cxk.head(dc.TOPK_ACT2).iterrows():
        number += 1; PICKS.append(pick_row(number, "Act 2", key, c, namesk))
# Act 1 by climate future: clusters on the balanced scenario at each refugia realization (own numbering "585-1..", "245-1..")
for lv, F in F_LEV.items():
    labL, regL = dc.clusters(G, F, min_km2=MIN_KM2)
    regL.insert(0, "act", f"Act 1 ({lv})"); regL.insert(1, "key", f"core_{lv}")
    regL["name"] = [dc.placeholder_name(G, labL == c, named) if k else "" for c, k in zip(regL.cid, regL.kept)]
    regL, cxL = dc.group_complexes(G, labL, regL, link_km=COMPLEX_LINK_KM)
    LABELS[f"act1_{lv}"] = labL; REGS.append(regL); CX[f"core_{lv}"] = cxL
    sens.append(dc.sensitivity(G, F, min_km2=MIN_KM2).assign(act=f"Act 1 ({lv})", key=f"core_{lv}"))
    namesL = dict(zip(regL.cid.astype(int), regL["name"]))
    for j, (_, c) in enumerate(cxL.head(dc.TOPK_ACT1).iterrows(), 1):
        PICKS.append(pick_row(f"{lv}-{j}", f"Act 1 ({lv})", f"core_{lv}", c, namesL))
    print(f"Act 1 ({lv}): tier {int(((F >= THR) & G.disc).sum()):,} km2 -> {int(regL.kept.sum())} clusters >= {MIN_KM2} km2")
REG = pd.concat(REGS, ignore_index=True); REG.to_csv(TAB / "cluster_register_all.csv", index=False)
SENS = pd.concat(sens, ignore_index=True); SENS.to_csv(TAB / "cluster_sensitivity.csv", index=False)
# ---- deck picks, second presentational tier (parent M4.33): REGIONAL clusters (single linkage at PICK_LINK_KM) numbered NORTH -> SOUTH;
# the core first (1..n), then each scenario's picks continue the numbering; by-future picks keep their own labels
RAW_PICKS = pd.DataFrame(PICKS); RAW_PICKS.to_csv(TAB / "picks_raw_complexes.csv", index=False)
grouped, number = [], 0
PICK_COLS = ["act", "key", "cid", "cids", "n_components", "name", "km2", "meanF", "lat", "lon", "members"]
core_raw = RAW_PICKS[RAW_PICKS.act == "Act 1"] if len(RAW_PICKS) else pd.DataFrame(columns=PICK_COLS[:-1] + ["number"])
if len(core_raw):
    gp_ = dc.group_picks(G, LABELS["act1"], core_raw, link_km=PICK_LINK_KM)
    gp_ = dc.absorb_complexes(G, LABELS["act1"], gp_, cx1, link_km=PICK_LINK_KM, reg=reg1, speck_km=SPECK_LINK_KM)   # nearby complexes AND sub-floor specks join the nearest regional cluster (M4.33 i, k)
    if MAX_CORE_CLUSTERS_AB and len(gp_) > MAX_CORE_CLUSTERS_AB:      # deck picks = clusters 1..MAX_CORE_CLUSTERS_AB in the north -> south numbering (parent M4.40 addendum; None = all)
        _drop = gp_.iloc[MAX_CORE_CLUSTERS_AB:]
        print(f"core picks trimmed to clusters 1..{MAX_CORE_CLUSTERS_AB} (north -> south); dropped: " + "; ".join(f"{r.name} ({r.km2:,.0f} km2, {r.lat:.1f}N)" for r in _drop.itertuples()))
        gp_ = gp_.iloc[:MAX_CORE_CLUSTERS_AB]
    if MERGE_CORE_PICKS:
        gp_ = gp_.reset_index(drop=True); gp_["_n0"] = gp_.index + 1   # the pre-merge north -> south number each pair refers to
        keep = []; used = set()
        for pair in MERGE_CORE_PICKS:
            mem = gp_[gp_._n0.isin(pair)]
            assert len(mem) == len(pair) and not (set(pair) & used), f"MERGE_CORE_PICKS {pair}: numbers must exist and be distinct across pairs (have 1..{len(gp_)})"
            big = mem.loc[mem.km2.idxmax()]; km2 = float(mem.km2.sum()); used |= set(pair)
            keep.append(dict(act=big.act, key=big.key, cid=int(big.cid), cids=";".join(mem.cids.astype(str)), n_components=int(mem.n_components.sum()), name=big["name"], km2=km2,
                             meanF=float((mem.meanF * mem.km2).sum() / km2), lat=float((mem.lat * mem.km2).sum() / km2), lon=float((mem.lon * mem.km2).sum() / km2),
                             members="+".join(f"[{m.members}]" for m in mem.itertuples())))
            print(f"merged core clusters {pair} -> {big['name']} ({km2:,.0f} km2)")
        rest = gp_[~gp_._n0.isin(used)].drop(columns="_n0")
        gp_ = pd.concat([rest, pd.DataFrame(keep)], ignore_index=True).sort_values("lat", ascending=False).reset_index(drop=True)
else:
    gp_ = pd.DataFrame(columns=PICK_COLS); print("NO core clusters >= MIN_KM2 at the applied band -- the core is (nearly) empty on this block (H-AB4 outcome); Act 1 has no picks")
for _, r in gp_.iterrows():
    number += 1; grouped.append(dict(number=str(number), **r.to_dict()))
print(f"core picks: {len(core_raw)} complexes -> {len(gp_)} regional clusters (single linkage {PICK_LINK_KM} km), numbered north -> south: "
      + "; ".join(f"{i + 1} = {r.name} ({r.km2:,.0f} km2; from picks {r.members})" for i, r in enumerate(gp_.itertuples())))
for key in POOL:
    sc_raw = RAW_PICKS[(RAW_PICKS.act == "Act 2") & (RAW_PICKS.key == key)] if len(RAW_PICKS) else RAW_PICKS
    if not len(sc_raw):
        continue
    gs = dc.group_picks(G, LABELS[f"act2_{key}"], sc_raw, link_km=PICK_LINK_KM)
    for _, r in gs.iterrows():
        number += 1; grouped.append(dict(number=str(number), **r.to_dict()))
grouped += [dict(members=str(r["number"]), **{k: v for k, v in r.items()}) for r in PICKS if str(r["act"]).startswith("Act 1 (")]
PICKS = pd.DataFrame(grouped, columns=["number"] + PICK_COLS) if not grouped else pd.DataFrame(grouped); PICKS.to_csv(TAB / "picks.csv", index=False)
np.savez_compressed(GEO / "cluster_labels.npz", **LABELS)
print("\ndeck picks (regional clusters, numbered north -> south; the register ships in full):"); print(PICKS.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))
gp = GEO / "clusters.gpkg"
if gp.exists(): gp.unlink()
for k, lab in LABELS.items():
    keyname = "core" if k == "act1" else (f"core_{k[5:]}" if k.startswith("act1_") else k.replace("act2_", ""))
    picked = set(int(x) for _, r in PICKS[PICKS.key == keyname].iterrows() for x in str(r.cids).split(";") if str(x).strip()) if len(PICKS) else set()
    regk = REG[(REG.key == keyname) & (REG.kept | REG.cid.isin(picked))]        # kept components + the specks a cluster absorbed
    if len(regk):
        v = dc.vectorize(G, lab, regk.cid.tolist(), simplify_m=500).merge(regk[["cid", "name", "km2", "meanF"]], on="cid")
        v.to_file(gp, layer=k, driver="GPKG")
print(f"wrote {gp.relative_to(ROOT)} ({len(LABELS)} layers; 500 m simplification)")


Act 1 (balanced core): tier 2,141 km2 -> 121 components, 24 >= 10 km2 (2,238 km2)
Act 1 complexes (single linkage 10 km): 8 from 24 components; top 6: 1,625 km2 (12 comp.), 197 km2 (2 comp.), 167 km2 (3 comp.), 96 km2 (3 comp.), 61 km2 (1 comp.), 44 km2 (1 comp.)
Act 2 s1   (Climate refugia forward): 74 components, 8 kept after core subtraction (928 km2 residual) -> 5 complexes
Act 2 s2   (Structural connectivity forward): 163 components, 16 kept after core subtraction (1,208 km2 residual) -> 11 complexes
Act 2 s2c  (Climate corridors forward): 82 components, 2 kept after core subtraction (26 km2 residual) -> 2 complexes
Act 2 s3   (Mammal + bird richness forward): 91 components, 2 kept after core subtraction (42 km2 residual) -> 2 complexes
Act 2 s4   (Biomass + soil carbon forward): 130 components, 7 kept after core subtraction (268 km2 residual) -> 4 complexes
Act 1 (585): tier 2,649 km2 -> 20 clusters >= 10 km2
Act 1 (245): tier 2,030 km2 -> 30 clusters >= 10 km2
merged core cluste

In [5]:
# ---- T-D1 register: AB-local block percentiles (naturalness axis; representativeness = EFG-count percentile), drivers, AOIs, tenure, T-D7 ----
# denominators = ALBERTA's allocatable land (as the Alberta run's package, so the two decks read side by side); the parent's own deck scores Y2Y-wide
VALS = {f: np.nan_to_num(lc._read(AB / f"{f}.tif")[G.pu], nan=0.0) for f in lc.continuous_features()}
pct = {}
for f in sorted({x for d in dc.BLOCK_AXES.values() for x in d}):
    ref = np.sort(VALS[f][G.disc]); pct[f] = (np.searchsorted(ref, VALS[f], side="right") / len(ref)).astype(np.float32)
axes = {ax: sum(w * pct[f] for f, w in members.items()).astype(np.float32) for ax, members in dc.BLOCK_AXES.items()}
efg_paths = lc.efg_paths(AB)
efg = np.stack([np.nan_to_num(lc._read(p)[G.pu], nan=0.0) > 0 for p in efg_paths])
count = efg.sum(axis=0).astype(np.float32); ref = np.sort(count[G.disc])
axes["representativeness"] = (np.searchsorted(ref, count, side="right") / len(ref)).astype(np.float32)
P = SimpleNamespace(axes=axes, pct=pct, efg=efg, efg_names=[p.stem for p in efg_paths], efg_count=count)
theta = config.AUDIT["theta"]
DM = {"m_soc theta-tail": VALS["irrecoverable_carbon_m_soc"] >= theta * VALS["irrecoverable_carbon_m_soc"].mean()}
DM["connectivity spike (top 0.2%)"] = VALS["transboundary_connectivity"] >= np.quantile(VALS["transboundary_connectivity"], 0.998)
k = int(DM["m_soc theta-tail"].sum()); refu = VALS["climate_type_macrorefugia"]
dens = np.zeros(G.n_pu, bool); dens[np.argsort(refu, kind="stable")[-k:]] = True   # exactly k cells (the v4 velocity floor can tie at the top; parent M4.39 h)
DM["refugia densest (area-matched to the m_soc tail)"] = dens
rare = np.zeros(G.n_pu, bool); rarest = np.zeros(G.n_pu, bool); n_rare = n_rarest = 0
WINF = REC / "efg_window_footprints.csv"          # v3.1: rarity judged in the Alberta +250 km window (D-AB12; parent M4.29)
if WINF.exists():
    _w = pd.read_csv(WINF).set_index("feature"); _rare_set = set(_w.index[_w.rare_window])
    rare_key = f"rarest-EFG footprint (rare in the extent+{config.EFG_TARGET_WINDOW_KM} km window: <= {100 * config.EFG_RARE_WINDOW_PCT:g}% of the window)"
else:
    _rare_set = None; rare_key = f"rarest-EFG footprint (<= {100 * dc.RARE_EFG_PCT:g}% of PU each)"      # v1 rule
for i, p in enumerate(efg_paths):
    e = np.nan_to_num(lc._read(p)[G.pu], nan=0.0)
    if lc.leverage_of(e)[1] >= config.AUDIT["rare_cap"]: rare |= e > 0; n_rare += 1
    if ((p.stem in _rare_set) if _rare_set is not None else (efg[i].sum() <= dc.RARE_EFG_PCT * G.n_pu)): rarest |= efg[i]; n_rarest += 1
DM["rare-attainable EFG footprint"] = rare; DM[rare_key] = rarest
# the necessity test (E19, the PARENT's 15b): adequacy-forced cells Y2Y-wide (capture 1.0 in every member of every design formulation), read on the Alberta PU
E19_TIF = RUNS / "e19_forced.tif"
if E19_TIF.exists():
    with rasterio.open(E19_TIF) as _s: FORCED = _s.read(1)[G.pu]
    with rasterio.open(RUNS / "e19_forced_class.tif") as _s: FORCED_CLS = _s.read(1)[G.pu]
    FORCED_NAMES = {int(k): v for k, v in json.loads((dc.SPEC_REC / "E19_forced_classes.json").read_text()).items()}
    print(f"E19 forced layer loaded: {int((FORCED == 2).sum()):,} km2 forced in all formulations, {int((FORCED >= 1).sum()):,} in >= 1")
else:
    FORCED = None; print("E19 forced layer absent (the parent's 12c + 15b) -- adequacy columns will be NaN")
print(f"driver masks: m_soc theta-tail {k:,} cells | refugia densest (same area) | spike {int(DM['connectivity spike (top 0.2%)'].sum()):,} | "
      f"rare-attainable EFG footprint {int(rare.sum()):,} ({100 * rare.mean():.0f}% of PU) from {n_rare}/{N_EFG} | rarest {int(rarest.sum()):,} from {n_rarest}")
with rasterio.open(DATA / "derived" / "tenure_class.tif") as s: ten = s.read(1)[G.pu]
COEX = dc.coexistence_layer(G)                                   # naming only (parent package spec v1.12 decision e): the communities tessellation covers Alberta's census divisions
print("region lookup:", f"{COEX.source} ({len(COEX.gdf)} units)" if COEX is not None else "ABSENT -- run analyses/communities/02_region_names; labels fall back to the landmark placeholder")
# bear coexistence programs (parent M4.40 addendum, Ethan 2026-09-30, consequences row): the mean number of coexistence groups recorded in the
# census divisions an area overlaps, cell-weighted; units with no recorded group (NA) are EXCLUDED, never recoded to 0 (the 2026-09-16 NA rule)
_NG = np.full(len(COEX.gdf) + 1, np.nan) if COEX is not None else None
if COEX is not None:
    _NG[1:] = pd.to_numeric(COEX.gdf["n_groups"], errors="coerce").values
def bear_programs(mask2d):
    if COEX is None:
        return np.nan, 0
    v = _NG[COEX.zones[mask2d]]; ok = np.isfinite(v)
    return (float(v[ok].mean()) if ok.any() else np.nan), int(len(np.unique(COEX.zones[mask2d][COEX.zones[mask2d] > 0])))
_bp = lambda m1: dict(zip(("bear_programs_mean", "bear_units_overlapped"), bear_programs(dc.to_grid(G, m1, fill=False, dtype=bool))))
near_pa = ndimage.distance_transform_edt(~G.locked2d) <= 5
KEY2LAB = {"core": "act1", **{f"core_{lv}": f"act1_{lv}" for lv in F_LEV}, **{k_: f"act2_{k_}" for k_ in POOL}}
KEY2ACT = {"core": "Act 1", **{f"core_{lv}": f"Act 1 ({lv})" for lv in F_LEV}, **{k_: "Act 2" for k_ in POOL}}
VR = dc.ValueRatios(G, P)      # consequences: mean value in the cluster / mean over Alberta's allocatable land (the parent stack = the AB stack on this grid)
CXROWS = []
picked_cids = set(); PICK_MASKS = {}
for _, r in PICKS[~PICKS.act.str.startswith("Act 1 (")].iterrows():          # the deck picks (regional clusters) first
    cids = [int(c_) for c_ in str(r.cids).split(";")]; picked_cids |= {(r.key, c_) for c_ in cids}
    CXROWS.append(dict(act=r.act, key=r.key, cid=int(r.cid), cids=cids, name=r["name"]))
for key, cx in CX.items():                                                       # then every other kept complex
    regk = REG[REG.key == key].set_index("cid")
    for _, c in cx.iterrows():
        if any((key, int(x)) in picked_cids for x in c.cids):
            continue
        CXROWS.append(dict(act=KEY2ACT[key], key=key, cid=int(c.anchor_cid), cids=list(c.cids), name=regk.loc[int(c.anchor_cid), "name"]))
rows = []
for r in pd.DataFrame(CXROWS).itertuples():
    lab = LABELS[KEY2LAB[r.key]]; m2 = np.isin(lab, r.cids)
    if r.act == "Act 2": m2 = m2 & ~core2d
    m1 = m2[G.pu]; n = int(m1.sum())
    prof = dc.star_profile(P, m1)
    pick = PICKS[(PICKS.act == r.act) & (PICKS.key == r.key) & (PICKS.cid == r.cid)]
    row = dict(number=str(pick.number.iloc[0]) if len(pick) else "", name=r.name, act=r.act, n_components=len(r.cids),
               driving=r.key if r.act == "Act 2" else "balanced scenario (both futures)", area_km2=n * G.cell_km2,
               mean_guarded_F=float(F0[m1].mean()), min_guarded_F=float(F0[m1].min()),       # on THE core surface (balanced f)
               mean_ensemble_F=float(Fg[m1].mean()))                                        # the paper's F, beside it
    region, subregions, region_label = dc.region_of(COEX, m2)
    region_label = region                                  # Alberta: the region word only -- the lookup's Alberta sub-region field holds the principal-places description, not a short name (kept in `subregions`)
    if len(pick) and (r.act, str(pick.number.iloc[0])) in CLUSTER_REGION_OVERRIDES:
        region = region_label = CLUSTER_REGION_OVERRIDES[(r.act, str(pick.number.iloc[0]))]   # a per-cluster word (Division No. 15 spans the whole mountain strip); `region` is what the titles read (director_plot.cluster_label)
    row.update(region=region, subregions=subregions, region_label=region_label, **_bp(m1))
    if len(pick): PICK_MASKS[(r.act, str(pick.number.iloc[0]))] = m1
    row.update({f"pct_{a}": prof[a] for a in dc.STAR_AXES}); row["efg_classes_present"] = dc.efg_classes_present(P, m1)
    row.update({f"ratio_{a}": v for a, v in VR.of(m1).items()})           # consequences: x times the average allocatable cell
    row.update({f"driver_{k_}": 100 * float(v[m1].mean()) for k_, v in DM.items()})
    if FORCED is not None:
        fa = 100 * float((FORCED[m1] == 2).mean()); fy = 100 * float((FORCED[m1] >= 1).mean())
        cls = FORCED_CLS[m1 & (FORCED >= 1)]
        pin = FORCED_NAMES.get(int(np.bincount(cls).argmax()), "") if cls.size else ""
        row.update(pct_adequacy_forced=fa, pct_forced_any_formulation=fy, adequacy_pin=bool(fa >= 50), adequacy_pin_class=pin)
    else:
        row.update(pct_adequacy_forced=np.nan, pct_forced_any_formulation=np.nan, adequacy_pin=False, adequacy_pin_class="")
    row.update(mean_lat=float(dc.latlon(G)[0][m1].mean()), pct_within_5km_of_PA=100 * float(near_pa[m2].mean()),
               pct_in_NFZ=100 * float(AOI_MASK["Upper Smoky Nature-First zone"][m2].mean()),
               pct_in_SRP_area=100 * float(AOI_MASK["Upper Smoky SRP planning area"][m2].mean()),
               pct_private_ranchland=100 * float((ten[m1] == 5).mean()), pct_crown=100 * float(np.isin(ten[m1], [2, 3]).mean()),
               n_formulations_frequent=int(sum(float(L.f_guard[fid][m1].mean() >= THR) >= 0.5 for fid in FORMS)))
    rows.append(row)
TD1 = pd.DataFrame(rows).sort_values(["act", "area_km2"], ascending=[True, False]).reset_index(drop=True)
TD1.insert(3, "act_v16", TD1.act.map(dc.ACT_DISPLAY).fillna(TD1.act))
TD1.insert(5, "driving_label", [dc.SCENARIO_LABEL[d.split("@")[0]] if d.split("@")[0] in dc.SCENARIO_LABEL else d for d in TD1.driving])
TD1.to_csv(TAB / "T-D1_cluster_register.csv", index=False)
if "region_label" in TD1 and TD1.region_label.astype(bool).any():                    # carry the region words onto the picks record too
    _lab = TD1[TD1.number.astype(str).ne("")][["act", "number", "region", "region_label"]].astype({"number": str})
    PICKS = PICKS.drop(columns=[c for c in ("region", "region_label") if c in PICKS]).merge(_lab, on=["act", "number"], how="left"); PICKS.to_csv(TAB / "picks.csv", index=False)
show = ["number", "name", "region_label", "act_v16", "n_components", "driving_label", "area_km2", "mean_guarded_F", "mean_lat", "pct_in_NFZ", "pct_in_SRP_area",
        "pct_private_ranchland", "n_formulations_frequent", "pct_adequacy_forced", "adequacy_pin_class"] + [c for c in TD1.columns if c.startswith("driver_")]
if FORCED is not None:
    pins = TD1[TD1.adequacy_pin]
    print(f"ADEQUACY PINS (>= 50% of the cluster forced in every formulation): {len(pins)} of {len(TD1)} clusters -- "
          + (", ".join(f"{r['name']} ({r.adequacy_pin_class})" for _, r in pins.iterrows()) if len(pins) else "none"))
print(f"T-D1 (deck picks = regional clusters within {PICK_LINK_KM} km, then the other complexes within {COMPLEX_LINK_KM} km; % columns are shares of cells):")
print(TD1[show].to_string(index=False, float_format=lambda v: f"{v:,.1f}"))
# ---- T-D7 consequences (parent M4.33): the deck clusters' ratios + two REFERENCE rows on the same allocatable-land denominator:
# existing protected areas, and the Upper Smoky Nature-First zone's unprotected part (the IPCA analogue; the SRP planning area beside it)
RAT = TD1[TD1.act.isin(["Act 1", "Act 2"]) & (TD1.number != "")][["number", "name", "act", "driving_label", "area_km2", "mean_guarded_F", "bear_programs_mean", "bear_units_overlapped"] + [f"ratio_{a}" for a in dc.STAR_AXES]].copy()
RAT["number"] = RAT.number.astype(int); RAT = RAT.sort_values("number")
# reference columns (parent M4.40 addendum, Ethan 2026-09-23): NAMED example areas are the tables' reference columns (act "reference") --
# on Alberta: Banff National Park (existing PA, whole footprint) and the Upper Smoky Nature-First zone (the IPCA analogue, whole footprint
# on the PU); the three aggregates of M18.3 (all existing PAs; the Nature-First zone's and the SRP planning area's unprotected parts) stay in
# the CSV as act "reference_aggregate" (director_plot.STYLE["conseq_reference"] switches back to them)
def _ref(name, m1, act="reference_aggregate"):
    return dict(number=np.nan, name=name, act=act, driving_label="", area_km2=int(m1.sum()) * G.cell_km2, **_bp(m1),
                mean_guarded_F=(float(F0[m1 & G.disc].mean()) if (m1 & G.disc).any() else np.nan),
                pct_protected=100 * float(G.locked[m1].mean()), **{f"ratio_{a}": v for a, v in VR.of(m1).items()})
NFZ_ADD = AOI_MASK["Upper Smoky Nature-First zone"][G.pu] & ~G.locked
SRP_ADD = AOI_MASK["Upper Smoky SRP planning area"][G.pu] & ~G.locked
CONSEQ_REFERENCE_AREAS_AB = [("Willmore Wilderness Park", "pa", "Willmore"), ("Peter Lougheed Provincial Park", "pa", "Peter Lougheed")]   # Ethan 2026-09-30 (was Banff NP + the Upper Smoky Nature-First zone; those stay as aggregate rows in the CSV)
REF_MASKS = {}
for _label, _kind, _match in CONSEQ_REFERENCE_AREAS_AB:
    if _kind == "pa":
        _g = pa[pa["PA_Name"] == _match]; assert len(_g), f"reference area {_match!r} not found in the PA layer on the Alberta extent"
        _m1 = rfeatures.rasterize(((geom, 1) for geom in _g.geometry), out_shape=G.shape, transform=G.transform, fill=0, dtype="uint8").astype(bool)[G.pu]
    else:
        _m1 = AOI_MASK[_match][G.pu]
    REF_MASKS[_label] = _m1
    print(f"reference area {_label}: {int(_m1.sum()):,} km2 on the PU, {100 * G.locked[_m1].mean():.0f}% already protected")
REF = pd.DataFrame([_ref("Existing protected areas", G.locked), _ref("Upper Smoky Nature-First zone (unprotected part)", NFZ_ADD),
                    _ref("Upper Smoky SRP planning area (unprotected part)", SRP_ADD)] + [_ref(_label, _m1, act="reference") for _label, _m1 in REF_MASKS.items()])
RAT = pd.concat([RAT, REF], ignore_index=True); RAT.to_csv(TAB / "T-D7_consequences.csv", index=False)
# package spec v1.14 (parent M4.39 d): ratios are MEAN-to-mean. One-time check, not shown in the deck: the median-based ratio beside it
# for every cluster x value; flag pairs disagreeing by > 1.5x (expected: carbon -- concentrated and real, kept; refugia -- native on the floored layer under v4)
BASE_MED = {f: float(np.median(v[G.disc])) for f, v in VR.raw.items()}
def median_ratio(mask1d):
    out = {ax: float(sum(w * (np.median(VR.raw[f][mask1d]) / BASE_MED[f] if BASE_MED[f] > 0 else np.nan) for f, w in members.items()))
           for ax, members in dc.BLOCK_AXES.items()}
    out["representativeness"] = float(np.median(VR.raw["__efg_count"][mask1d]) / BASE_MED["__efg_count"]) if BASE_MED["__efg_count"] > 0 else np.nan
    return {a: out[a] for a in dc.STAR_AXES}
MM = {**{f"Cluster {n} ({a})": m for (a, n), m in PICK_MASKS.items()}, "Existing protected areas": G.locked, "Upper Smoky Nature-First zone (unprotected part)": NFZ_ADD, **REF_MASKS}
mm_rows = []
for nm, m in MM.items():
    mean_r, med_r = VR.of(m), median_ratio(m)
    for a in dc.STAR_AXES:
        q = mean_r[a] / med_r[a] if med_r[a] and np.isfinite(med_r[a]) and med_r[a] > 0 else np.nan
        mm_rows.append(dict(unit=nm, value=a, mean_ratio=mean_r[a], median_ratio=med_r[a], mean_over_median=q, flag=bool(np.isfinite(q) and (q > 1.5 or q < 1 / 1.5))))
MMC = pd.DataFrame(mm_rows); MMC.to_csv(TAB / "T-D7_mean_vs_median_check.csv", index=False)
flags = MMC[MMC.flag]
print(f"\nT-D7 mean-vs-median check (v1.14): {len(flags)} of {len(MMC)} cluster x value pairs disagree by > 1.5x"
      + (":\n" + flags[["unit", "value", "mean_ratio", "median_ratio", "mean_over_median"]].to_string(index=False, float_format=lambda v: f"{v:.2f}") if len(flags) else ""))
print("\nT-D7 consequences (mean value in the cluster / mean over allocatable land; reference rows italic in the render):")
print(RAT.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))


E19 forced layer loaded: 0 km2 forced in all formulations, 0 in >= 1
driver masks: m_soc theta-tail 4,384 cells | refugia densest (same area) | spike 171 | rare-attainable EFG footprint 39,914 (47% of PU) from 9/13 | rarest 247 from 1
region lookup: region_lookup.csv (108 units)
ADEQUACY PINS (>= 50% of the cluster forced in every formulation): 0 of 46 clusters -- none
T-D1 (deck picks = regional clusters within 20 km, then the other complexes within 10 km; % columns are shares of cells):
number                                                              name              region_label             act_v16  n_components                    driving_label  area_km2  mean_guarded_F  mean_lat  pct_in_NFZ  pct_in_SRP_area  pct_private_ranchland  n_formulations_frequent  pct_adequacy_forced adequacy_pin_class  driver_m_soc theta-tail  driver_connectivity spike (top 0.2%)  driver_refugia densest (area-matched to the m_soc tail)  driver_rare-attainable EFG footprint  driver_rarest-EFG footprint 

In [6]:
# ---- T-D3 (captures of the Y2Y-wide anchors WITHIN ALBERTA), tier achievement, T-D5 protected baseline, T-D5b enrichment, T-D4 Natural Regions ----
# capture here = the share of ALBERTA's total of each value held by the Y2Y-wide plan's Alberta cells (locked PAs included) -- the Alberta view of a
# corridor-wide plan; the plans' Y2Y-wide captures (what they were solved for) are the parent's T1 record (analyses/y2y/spec/v4/T1_anchor_captures.csv)
A_mat = np.stack([L.anchors[f] for f in FORMS])
cap = pd.DataFrame({f: (A_mat @ v) / v.sum() for f, v in VALS.items()}, index=FORMS)
theta_ = config.AUDIT["theta"]; tail = {}
for f, v in VALS.items():
    tl = v >= theta_ * v.mean(); tail[f] = (A_mat[:, tl] @ v[tl]) / v[tl].sum() if tl.any() else np.full(len(FORMS), np.nan)
tail = pd.DataFrame(tail, index=FORMS)
cap.to_csv(TAB / "T1_anchor_captures_within_alberta.csv"); tail.to_csv(TAB / "T1_tail_capture_within_alberta.csv")
lat, _ = dc.latlon(G)
rows = []
for _, r in MAN.iterrows():
    fid = r.formulation_id
    row = dict(formulation=fid, scenario=dc.SCENARIO_LABEL[r.scenario_id], climate=r.climate_level.replace("_2071_2100", ""), value_statement=dc.SCENARIO_STATEMENT[r.scenario_id])
    for b, feats in BLOCKS.items(): row[f"capture_{b}"] = float(cap.loc[fid, feats].mean())
    row["tail_m_soc"] = float(tail.loc[fid, "irrecoverable_carbon_m_soc"]); row["tail_biomass"] = float(tail.loc[fid, "irrecoverable_carbon_biomass"])
    row["anchor_mean_lat"] = float(lat[L.anchors[fid] & G.disc].mean())
    row["frequent_km2_guarded"] = int((L.f_guard[fid][G.disc] >= THR).sum()); row["frequent_km2_unguarded"] = int((L.f_plain[fid][G.disc] >= THR).sum()) if fid in L.f_plain else np.nan   # v4: the reference cell only
    row["D_unguarded"], row["D_guarded"] = L.D_plain.get(fid, np.nan), L.D_guard[fid]
    rows.append(row)
TD3 = pd.DataFrame(rows); TD3.to_csv(TAB / "T-D3_scenarios.csv", index=False)
print("T-D3:"); print(TD3.drop(columns="value_statement").to_string(index=False, float_format=lambda v: f"{v:,.3f}"))

core1, sc1, opp1 = core_m, AG["Act 2 any named scenario (union)"], AG["Act 3 opportunity (in >= 1 band, not above)"]
CUM = {"existing PAs": G.locked, "+ Act 1 core": G.locked | core1, "+ Act 2 scenario tiers": G.locked | core1 | sc1, "+ Act 3 opportunity": G.locked | core1 | sc1 | opp1}
rows = []
for t, m in CUM.items():
    for b, feats in BLOCKS.items():
        caps = [float(VALS[f][m].sum() / VALS[f].sum()) for f in feats]
        for f, c_ in zip(feats, caps): rows.append(dict(tier=t, block=b, feature=f, capture=c_))
        rows.append(dict(tier=t, block=b, feature="BLOCK", capture=float(np.mean(caps))))
TA = pd.DataFrame(rows)
ref = TD3[[c for c in TD3.columns if c.startswith("capture_")]].rename(columns=lambda c: c.replace("capture_", "")); ref.index = TD3.formulation
TA_ref = pd.DataFrame({"s0": ref.loc["s0_ssp585_theta5"], "anchor_min": ref.min(), "anchor_max": ref.max()})
TA.to_csv(TAB / "tier_achievement.csv", index=False); TA_ref.to_csv(TAB / "tier_achievement_reference.csv")
print("\ntier achievement (block capture, cumulative tiers incl. locked PAs):")
print(TA[TA.feature == "BLOCK"].pivot(index="tier", columns="block", values="capture").reindex(list(CUM)).to_string(float_format=lambda v: f"{v:.3f}"))
SUMMARY["tier_area_pct_disc"] = {k_: float(100 * (m & G.disc).sum() / G.n_disc) for k_, m in [("core", core1), ("scenario", sc1), ("opportunity", opp1)]}

# T-D5: what the existing estate already banks; S0_target = the Y2Y-WIDE target the plans were solved under (an amount of the Y2Y total, not of Alberta's --
# "% of target banked" is therefore indicative only); "new half" = the reference plan's Alberta additions
sc0 = SC["S0_balanced"]["targets"]; pa_share = float(G.locked.sum() / G.n_pu); add_share = BUDGET_PCT_A - pa_share
rows = []
for f, v in VALS.items():
    pa_ = float(v[G.locked].sum() / v.sum()); tgt = float(sc0.get(f, 1.0)); s0cap = float(cap.loc["s0_ssp585_theta5", f])
    rows.append(dict(value=f, pct_of_extent_total_in_PAs=100 * pa_, S0_target=tgt, pct_of_target_already_banked=100 * pa_ / tgt,
                     pct_still_needed_from_unprotected_land=100 * max(tgt - pa_, 0), enrichment_existing_PAs=pa_ / pa_share,
                     enrichment_S0_new_half=(s0cap - pa_) / add_share))
TD5 = pd.DataFrame(rows); TD5.to_csv(TAB / "T-D5_protected_baseline.csv", index=False)
n_efg_pa = int(P.efg[:, G.locked].any(axis=1).sum())
SUMMARY["protected_baseline"] = dict(pa_km2=int(G.locked.sum()), pa_pct_of_region=float(100 * pa_share), pa_pct_of_budget=float(100 * pa_share / BUDGET_PCT_A),
                                     efg_present_in_PAs=n_efg_pa, banked_min=float(TD5.pct_of_extent_total_in_PAs.min()), banked_max=float(TD5.pct_of_extent_total_in_PAs.max()))
print("\nT-D5 (existing PAs, locked in; enrichment = capture share / area share; 'new half' = the Y2Y-wide reference plan's Alberta additions; S0_target = Y2Y-wide):")
print(TD5.to_string(index=False, float_format=lambda v: f"{v:.1f}"))
banked = {f: float(v[G.locked].sum() / v.sum()) for f, v in VALS.items()}
def tier_enrich(m1):
    a = m1.sum() / G.n_pu
    return {f: float(v[m1].sum() / v.sum()) / a if a > 0 else np.nan for f, v in VALS.items()}
E5 = {}
for _, r in MAN.iterrows():
    fid = r.formulation_id; lab_ = f"{dc.SCENARIO_LABEL[r.scenario_id].split(' (')[0]} {'585' if 'ssp585' in fid else '245'}"
    E5[f"anchor - {lab_}"] = {f: (float(cap.loc[fid, f]) - banked[f]) / add_share for f in VALS}
    E5[f"frequent tier - {lab_}"] = tier_enrich((L.f_guard[fid] >= THR) & G.disc)
E5["frequent tier - BALANCED core (applied)"] = tier_enrich(core1); E5["frequent tier - ENSEMBLE core (paper)"] = tier_enrich((Fg >= THR) & G.disc)
E5["existing PAs"] = {f: banked[f] / pa_share for f in VALS}
TD5b = pd.DataFrame(E5); TD5b.index.name = "value"; TD5b.to_csv(TAB / "T-D5b_enrichment_by_scenario.csv")

# T-D4: Natural Regions / Subregions of Alberta (2005) -- the ecoregion analogue (acquired; REST layer 0)
NSR = gpd.read_file(DATA / "aoi" / "natural_subregions_2005.gpkg").to_crs(G.crs)
zones_id = rfeatures.rasterize(((g, i + 1) for i, g in enumerate(NSR.geometry)), out_shape=G.shape, transform=G.transform, fill=0, dtype="int32")[G.pu]
rows = []
for i, (nr, ns) in enumerate(zip(NSR.NRNAME, NSR.NSRNAME), start=1):
    inz = zones_id == i
    if not inz.any(): continue
    rows.append({"natural_region": nr, "subregion": ns, "PU km2": int(inz.sum()), "protected km2": int((inz & G.locked).sum()),
                 "core km2": int((inz & (tiers == 3)).sum()), "scenario km2": int((inz & (tiers == 2)).sum()),
                 "opportunity km2": int((inz & (tiers == 1)).sum()), "never km2": int((inz & G.disc & (tiers == 0)).sum()), "mean lat": float(lat[inz].mean())})
TD4 = pd.DataFrame(rows).groupby(["natural_region", "subregion"]).agg({"PU km2": "sum", "protected km2": "sum", "core km2": "sum", "scenario km2": "sum",
                                                                          "opportunity km2": "sum", "never km2": "sum", "mean lat": "mean"}).reset_index().sort_values("core km2", ascending=False)
TD4.to_csv(TAB / "T-D4_natural_subregions.csv", index=False)
TD4r = TD4.groupby("natural_region").sum(numeric_only=True).drop(columns="mean lat").sort_values("core km2", ascending=False); TD4r.to_csv(TAB / "T-D4_natural_regions.csv")
SUMMARY["td4"] = "Natural Regions and Subregions of Alberta (2005), NRNAME / NSRNAME"
print("\nT-D4 by Natural Region:"); print(TD4r.to_string())


T-D3:
      formulation                        scenario climate  capture_core_habitat  capture_structural_connectivity  capture_climate_corridors  capture_carbon  capture_biodiversity  tail_m_soc  tail_biomass  anchor_mean_lat  frequent_km2_guarded  frequent_km2_unguarded  D_unguarded  D_guarded
 s0_ssp585_theta5                        Balanced  ssp585                 0.586                            0.762                      0.468           0.579                 0.428       0.898         0.646           52.210                  2649                  77.000        1.293      1.008
 s1_ssp585_theta5         Climate refugia forward  ssp585                 0.650                            0.771                      0.516           0.631                 0.477       0.980         0.767           52.023                  4097                     NaN          NaN      0.860
 s2_ssp585_theta5 Structural connectivity forward  ssp585                 0.564                            0.771         

In [7]:
# ---- Act 0 (where the values are), Act 3 (the measured gap), the hinge cross-tab; the E19 partition (parent 19 cell 8) -------------
# Value = the block percentile used for the star axes, thresholded at the top 30% of Alberta's DISCRETIONARY land;
# representativeness votes as presence of a class rare in the Alberta +250 km window (D-AB12; the rare-attainable set is
# reported, not used); naturalness (1 - gHM) is a further map, "disclosed, not a driver", outside the convergence count (0-5 up to v3.1; 0-6 under v4).
rare_key_ = [k for k in DM if k.startswith("rarest-EFG")][0]
V = dc.value_layers(G, P, DM[rare_key_])
for t, m in V.masks.items():
    dc.write_tif(G, m.astype(np.uint8), GEO / f"value_top30_{t.replace(' ', '_')}.tif", dtype="uint8", nodata=255)
dc.write_tif(G, V.convergence, GEO / "value_convergence.tif", dtype="uint8", nodata=255)
tier_code = tiers                                     # 0 never / 1 opportunity / 2 scenario / 3 core (cell 3)
high = (V.convergence >= 1) & G.disc
gap = high & (tier_code <= 1)                         # Act 3: high-value land that no tier makes irreplaceable
dc.write_tif(G, np.where(gap, V.convergence, 0).astype(np.uint8), GEO / "value_gap.tif", dtype="uint8", nodata=255)
TD6 = dc.coverage_table(G, V, tier_code)
alt = DM["rare-attainable EFG footprint"] & G.disc
TD6.loc[len(TD6)] = {"theme": f"representativeness — rare-attainable classes (NOT used)", "top-value footprint km2": int(alt.sum()),
                     "% of unprotected land": 100 * alt.sum() / G.n_disc,
                     **{f"% of footprint in {nm}": 100 * float((alt & (tier_code == code)).sum()) / max(alt.sum(), 1) for code, nm in dc.TIER_NAMES.items()}}
TD6.to_csv(TAB / "T-D6_value_coverage.csv", index=False)
print("T-D6 value coverage (top-30% footprint of each theme over unprotected land, and which reliability tier holds it):")
print(TD6.to_string(index=False, float_format=lambda v: f"{v:,.1f}"))
def _ta(masks):                                       # share of each block's Alberta value by tier (dc.tier_achievement reads the parent stack)
    rows_ = []
    for tier, m in masks.items():
        for b, feats in BLOCKS.items():
            caps_ = [float(VALS[f][m].sum() / VALS[f].sum()) for f in feats]
            for f, c_ in zip(feats, caps_): rows_.append(dict(tier=tier, block=b, feature=f, capture=c_))
            rows_.append(dict(tier=tier, block=b, feature="BLOCK", capture=float(np.mean(caps_))))
    return pd.DataFrame(rows_)
VS = _ta({"existing PAs": G.locked, "core": core1, "scenario tiers": sc1, "opportunity": opp1, "never": G.disc & (tier_code == 0)})
VSp = VS[VS.feature == "BLOCK"].pivot(index="tier", columns="block", values="capture").reindex(["existing PAs", "core", "scenario tiers", "opportunity", "never"])
VSp.to_csv(TAB / "T-D6b_value_share_by_tier.csv")
print("\nshare of each block's Alberta value by tier (rows sum to 1 over the extent):"); print(VSp.to_string(float_format=lambda v: f"{v:.3f}"))
XT = dc.crosstab(G, V.convergence, tier_code); XT.to_csv(TAB / "hinge_crosstab.csv")
print("\nhinge cross-tab (km2): value-convergence count x reliability class:"); print(XT.to_string())
conv_km2 = {k: int(((V.convergence == k) & G.disc).sum()) for k in range(len(dc.VALUE_THEMES) + 1)}
print(f"\nhigh-value land (>= 1 theme): {int(high.sum()):,} km2 = {100 * high.sum() / G.n_disc:.0f}% of unprotected land; "
      f"of it {int(gap.sum()):,} km2 ({100 * gap.sum() / max(high.sum(), 1):.0f}%) sits outside the core and the scenario tiers = Act 3")
# biodiversity: the finding is the product -- capture range over EVERY guarded plan at the applied band
bio = [VALS[f] for f in BLOCKS["biodiversity"]]; bio_tot = [float(v.sum()) for v in bio]
caps = []
for fid in FORMS:
    Sg = np.vstack([L.anchors[fid][None, :], ec.read_selections(RUNS / fid / f"mga_guard_{BAND}.tif", G.pu)])
    caps += [float(np.mean([v[row].sum() / t for v, t in zip(bio, bio_tot)])) for row in Sg]
    del Sg
caps = np.array(caps)
SUMMARY["biodiversity_plan_capture"] = dict(n_plans=int(caps.size), min=float(caps.min()), max=float(caps.max()), median=float(np.median(caps)))
print(f"biodiversity block capture over all {caps.size} guarded plans: {100 * caps.min():.1f}-{100 * caps.max():.1f}% (median {100 * np.median(caps):.1f}%)")
if FORCED is not None:
    conv_cont = sum(V.masks[t].astype(np.uint8) for t in dc.VALUE_THEMES if t != "representativeness")   # the non-EFG themes (4 before v4, 5 under v4)
    def partition(m):
        n = max(int(m.sum()), 1); f = m & (FORCED == 2); mc = m & ~f & (conv_cont >= 2)
        return dict(km2=int(m.sum()), forced_pct=100 * f.sum() / n, multi_claim_pct=100 * mc.sum() / n, other_pct=100 * (m & ~f & ~mc).sum() / n)
    PARTS = {"core": core1, **{f"scenario tier: {dc.SCENARIO_LABEL[s]}": (owner == i) for i, s in enumerate(dc.ACT2_SCENARIOS, 1)}, "opportunity": opp1}
    for _, r in PICKS.iterrows():
        lab = LABELS[KEY2LAB[r.key]]; m2 = np.isin(lab, [int(c_) for c_ in str(r.cids).split(";")])
        if r.act == "Act 2": m2 = m2 & ~core2d
        PARTS[f"pick {r.number}: {r['name']}"] = m2[G.pu]
    E19P = pd.DataFrame([{"unit": k_, **partition(m)} for k_, m in PARTS.items()])
    E19P.to_csv(TAB / "E19_partition.csv", index=False)
    print("\nthe necessity test (E19) partition -- forced / multi-claim (>= 2 non-EFG themes top-30%) / other:")
    print(E19P.to_string(index=False, float_format=lambda v: f"{v:,.1f}"))
    SUMMARY["e19"] = dict(core_forced_pct=float(E19P.iloc[0].forced_pct), n_pins=int(TD1.adequacy_pin.sum()), partition=E19P.to_dict(orient="records"))
SUMMARY["value"] = dict(top=V.top, rare_efg_rule=rare_key_, footprint_km2={t: int(m.sum()) for t, m in V.masks.items()},
                        convergence_km2=conv_km2, high_value_km2=int(high.sum()), gap_km2=int(gap.sum()),
                        gap_pct_of_high_value=float(100 * gap.sum() / max(high.sum(), 1)),
                        value_share_by_tier=VSp.to_dict(), coverage=TD6.to_dict(orient="records"), crosstab=XT.to_dict())


T-D6 value coverage (top-30% footprint of each theme over unprotected land, and which reliability tier holds it):
                                                  theme  top-value footprint km2  % of unprotected land  % of footprint in core  % of footprint in scenario tiers  % of footprint in opportunity  % of footprint in never
                                           core habitat                    17149                   30.0                    11.7                               9.1                           79.3                      0.0
                                structural connectivity                    17149                   30.0                    12.4                               8.7                           78.9                      0.0
                                      climate corridors                    17149                   30.0                     5.2                               3.2                           90.1                      1.5
              

In [8]:
# ---- T-D0: the objectives rows for THIS package (16 renders them through director_plot.values_table): the Y2Y-wide analysis as the
#      plans were solved (30% of the whole region, Y2Y-wide weights and targets, the 20-class block), with what that places in Alberta ----
t0 = SC["S0_balanced"]["targets"]; t4 = SC["S4_carbon"]["targets"]; MS = "irrecoverable_carbon_m_soc"
EFG_T = json.loads((dc.SPEC_REC / "efg_targets.json").read_text())["targets"]; pb = SUMMARY["protected_baseline"]; ADD = SUMMARY["additions_km2"]
ADD_LO, ADD_HI = SUMMARY["additions_km2_range"]
assert len(BLOCKS) == 5, "manifest v4 (five discretionary blocks at 20%) is the registered design"
VALUES_ROWS_Y = [
 ("PROTECT — wildlife have sufficient core habitat", "Quantity of core habitat", f"Protected land: today's protected areas are locked in ({pb['pa_pct_of_region']:.0f}% of the Alberta extent); the Y2Y-wide plans select 30% of the whole Y2Y region and place {ADD:,} km² of additions in Alberta (balanced plan; {ADD_LO:,}–{ADD_HI:,} km² across the 14 positions)",
  "Y2Y protected areas 2025 (IUCN definitions)", "The budget is Y2Y-wide (30% of the region); Alberta's share of the additions is an outcome of the corridor-wide optimum, not a budget", f"{ADD:,} km² of additions in Alberta (an outcome)"),
 ("", "Quality of core habitat", "Climate refugia: refugial residence time (1 / backward climate velocity, floored at one cell width over the horizon), 2071–2100, two emission futures",
  "AdaptWest 2023, CMIP6 backward climate velocity (8-GCM ensemble)", "Climate-refugia theme: 20% of the objective in the balanced position; the two futures are separate value positions", "20% of the objective"),
 ("", "Quality of core habitat", "Naturalness: 1 − human modification", "Theobald et al., global human modification (gHM v3)",
  "In every formulation at its baseline weight; it cannot move the answer — disclosed, not a driver", "baseline weight · not a driver"),
 ("", "Biodiversity", "Mammal richness (species per km², area-of-habitat maps, all species)", "Lumbierres et al., AOH species richness (mammals)",
  "Biodiversity theme: 20% (balanced); the two layers weighted equally", "10% of the objective"),
 ("", "Biodiversity", "Bird richness (species per km², area-of-habitat maps, all species)", "Lumbierres et al., AOH species richness (birds)", "", "10% of the objective"),
 ("", "Representativeness", f"Presence of {N_EFG_Y2Y} ecosystem functional groups across the Y2Y region ({N_EFG} of them present in Alberta), curated under the input pre-screen (rule R0)",
  "IUCN Global Ecosystem Typology, indicative maps (Keith et al. 2022)",
  f"A representation floor, not a weighted theme: rarity-scaled targets of {100*min(EFG_T.values()):.0f}–{100*max(EFG_T.values()):.0f}% per class, rarity judged in the Y2Y region + 250 km",
  f"targets {100*min(EFG_T.values()):.0f}–{100*max(EFG_T.values()):.0f}% per class"),
 ("CONNECT — wildlife corridors connect core habitats", "Structural connectivity", "Habitat connectivity: transboundary omnidirectional current density, valued convexly (current squared) — pinch points count more than proportionally",
  "Pither et al. 2023 / O'Brien et al. (transboundary extension)", "Structural-connectivity theme: 20% (balanced); no security target — pinch points are places, not an amount (the R2 semantic gate)", "20% of the objective"),
 ("", "Climate corridors", "Climate corridors: current-flow centrality across climate-analog classes", "Carroll et al. 2018",
  "Climate-corridors theme: 20% (balanced); its own forward position (S2c)", "20% of the objective"),
 ("ADDRESS CLIMATE CHANGE — keep carbon out of the air", "Carbon", "Irrecoverable carbon in biomass", "Berman & McDowell, irrecoverable carbon",
  "Carbon theme: 20% (balanced); the two pools split by mass across the Y2Y region", "carbon share · biomass"),
 ("", "Carbon", "Irrecoverable carbon in mineral soil", "Berman & McDowell, irrecoverable carbon",
  f"Security target: {100*t0[MS]:.0f}% of the Y2Y-wide total ({100*t4[MS]:.0f}% in the carbon-forward position); Alberta holds whatever share of it the corridor-wide plan takes here",
  f"carbon share · soil · target {100*t0[MS]:.0f}% ({100*t4[MS]:.0f}%) of the Y2Y total"),
 ("NOT IN THIS ANALYSIS", "Communities · Water · Cost · Tenure", "Bear-smart communities; water; dollars; crown vs private", "—",
  "Tenure is reported after the fact (crown → protection track, private ranchland → OECM track), never optimized; the Y2Y-wide area budget stands in for cost", "—"),
]
pd.DataFrame(VALUES_ROWS_Y, columns=["fundamental_objective", "sub_objective", "performance_measure", "source", "in_the_analysis", "metric"]).to_csv(TAB / "T-D0_values_rows.csv", index=False)   # = director_plot.VALUES_COLUMNS + metric
print(f"wrote {(TAB / 'T-D0_values_rows.csv').relative_to(ROOT)} ({len(VALUES_ROWS_Y)} rows; 16 renders it to the table spec)")


wrote analyses/alberta_prioritization/director_package_y2y_analysis/tables/T-D0_values_rows.csv (11 rows; 16 renders it to the table spec)


In [9]:
# ---- AOI / C4 summary for the deck + tenure by tier + summary.json ------------------------------------------------------
rows = []
for n, m2 in AOI_MASK.items():
    m = m2[G.pu] & G.disc
    rows.append(dict(aoi=n, km2_unlocked=int(m.sum()), share_of_unprotected_pct=100 * m.sum() / G.n_disc, core_inside_km2=int((core1 & m).sum()),
                     scenario_inside_km2=int((sc1 & m).sum()), opportunity_inside_km2=int((opp1 & m).sum()), mean_F_inside=float(Fg[m].mean()), mean_F_unprotected=float(Fg[G.disc].mean()),
                     **{f"mean_f_{k_}": float(POOL[k_][m].mean()) for k_ in POOL if k_ in dc.ACT2_SCENARIOS}))
TAOI = pd.DataFrame(rows); TAOI.to_csv(TAB / "T-AOI_alignment.csv", index=False)
print("AOI alignment (alignment, not assignment):"); print(TAOI.to_string(index=False, float_format=lambda v: f"{v:.2f}"))
NAMES = {2: "crown (Green Area)", 3: "crown (White Area, notated)", 4: "private presumed (non-ranch)", 5: "private ranchland (OECM track)", 6: "unclassified"}
TEN = pd.DataFrame([{"tier": lab_, "km2": int(((tiers == code_) & G.disc).sum()), **{nm: int(((tiers == code_) & G.disc & (ten == k_)).sum()) for k_, nm in NAMES.items()}}
                    for code_, lab_ in ((3, "core"), (2, "value-forward"), (1, "opportunity"), (0, "never"))])
TEN.to_csv(TAB / "T-TEN_tenure_by_tier.csv", index=False)
print("\ntenure by tier (km2; private classes over-counted by the crown-lease share, D-AB8):"); print(TEN.to_string(index=False))
SUMMARY["aoi"] = TAOI.to_dict(orient="records"); SUMMARY["tenure_by_tier"] = TEN.to_dict(orient="records")
# E17 leave-one-theme-out shifts on the curated block (11b's E17-T3 cell; parent M4.32) -- a T-D line, no one-pager (M11.4)
E17_DIR = RUNS / "e17_t3"; lat_, _ = dc.latlon(G)
if (E17_DIR / "efg_out" / "run" / "portfolio.tif").exists():
    s0 = L.anchors["s0_ssp585_theta5"] & G.disc; base_lat = float(lat_[s0].mean())
    rows_ = []
    for b in list(BLOCKS) + ["efg"]:                       # five block-outs + the EFG-out under v4 (six arms); four + EFG before
        p_ = E17_DIR / f"{b}_out" / "run" / "portfolio.tif"
        if not p_.exists(): continue
        sel = ec.read_selections(p_, G.pu)[0] & G.disc
        rows_.append(dict(block_out=b, mean_lat=float(lat_[sel].mean()), delta_lat=float(lat_[sel].mean() - base_lat), jaccard_vs_s0=dc.jaccard(sel, s0),
                          basis=f"{VERSION} (the parent's E17-T3 arms, Y2Y-wide S0; latitude of the Alberta cells)"))
    E17 = pd.DataFrame(rows_); E17.to_csv(TAB / "E17_shifts.csv", index=False)
    SUMMARY["e17"] = dict(base_lat=base_lat, basis=f"{VERSION}", shifts=E17.to_dict(orient="records"))
    print("E17 leave-one-theme-out (the parent's arms, read within Alberta):"); print(E17.to_string(index=False, float_format=lambda v: f"{v:+.3f}"))
else:
    print("E17-T3 anchors absent (the parent's 12c) -- no shifts recorded")
# ---- THE COMPARISON: this package (the Y2Y-wide analysis within Alberta) against the Alberta run's package (director_package/, notebook 12; R9.8) --
#      core areas, overlap, rank agreement of the two balanced f surfaces, each deck cluster's share inside the OTHER analysis's core, the tier cross-tab
if (PKG_AB / "summary.json").exists():
    S_AB = json.loads((PKG_AB / "summary.json").read_text())
    assert S_AB.get("version") == VERSION and S_AB.get("core_basis") == "balanced", f"the Alberta run's package is {S_AB.get('version')} / {S_AB.get('core_basis')} -- re-run 12 first"
    def _rd_ab(name):
        with rasterio.open(PKG_AB / "geotiffs" / name) as _s: return _s.read(1)[G.pu]
    F0_AB, Fg_AB, TIERS_AB, OWNER_AB = _rd_ab("f_balanced_core.tif"), _rd_ab("F_guarded.tif"), _rd_ab("act_tiers_guarded.tif"), _rd_ab("act2_owner.tif")
    core_ab = (F0_AB >= THR) & G.disc; core_y = core1; ens_ab = (Fg_AB >= THR) & G.disc; ens_y = (Fg >= THR) & G.disc
    from scipy import stats as _st
    def _pair(name, a, b):                     # a = the Alberta run, b = this (Y2Y-wide within Alberta)
        return dict(surface=name, alberta_run_km2=int(a.sum()), y2y_within_alberta_km2=int(b.sum()), shared_km2=int((a & b).sum()),
                    pct_of_alberta_core_in_y2y=100 * float((a & b).sum() / max(a.sum(), 1)), pct_of_y2y_core_in_alberta=100 * float((a & b).sum() / max(b.sum(), 1)),
                    jaccard=dc.jaccard(a, b))
    CMP = pd.DataFrame([_pair("balanced core (f >= 0.70)", core_ab, core_y), _pair("ensemble core (F >= 0.70, appendix)", ens_ab, ens_y),
                        _pair("always (balanced f >= 0.95)", (F0_AB >= 0.95) & G.disc, (F0 >= 0.95) & G.disc),
                        _pair("Act 2 scenario tiers (union)", (TIERS_AB == 2), (tiers == 2)), _pair("Act 3 opportunity", (TIERS_AB == 1), (tiers == 1))])
    CMP["spearman_f_balanced"] = [float(_st.spearmanr(F0_AB[G.disc], F0[G.disc]).correlation)] + [np.nan] * (len(CMP) - 1)
    CMP.to_csv(TAB / "comparison_vs_alberta_run.csv", index=False)
    # the deck clusters, each scored against the other analysis's core and surfaces
    LAB_AB = dict(np.load(PKG_AB / "geotiffs" / "cluster_labels.npz")); PK_AB = pd.read_csv(PKG_AB / "tables" / "picks.csv", dtype={"number": str, "cids": str})
    rows_ = []
    for _, r in PK_AB[PK_AB.act.isin(["Act 1", "Act 2"])].iterrows():
        m = np.isin(LAB_AB["act1" if r.act == "Act 1" else f"act2_{r.key}"], [int(c) for c in str(r.cids).split(";")])[G.pu] & G.disc
        if r.act == "Act 2": m = m & ~core_ab
        n = max(int(m.sum()), 1)
        rows_.append(dict(package="Alberta run", cluster=f"Cluster {r.number} ({r.act})", name=r["name"], km2=int(m.sum()), mean_f_own=float(F0_AB[m].mean()) if m.any() else np.nan,
                          pct_in_other_core=100 * float((m & core_y).sum()) / n, pct_in_other_scenario_tier=100 * float((m & (tiers == 2)).sum()) / n,
                          pct_in_other_opportunity=100 * float((m & (tiers == 1)).sum()) / n, mean_f_other=float(F0[m].mean()) if m.any() else np.nan))
    for _, r in PICKS[PICKS.act.isin(["Act 1", "Act 2"])].iterrows():
        m2_ = np.isin(LABELS[KEY2LAB[r.key]], [int(c) for c in str(r.cids).split(";")])
        if r.act == "Act 2": m2_ = m2_ & ~core2d
        m = m2_[G.pu] & G.disc; n = max(int(m.sum()), 1)
        rows_.append(dict(package="Y2Y-wide within Alberta", cluster=f"Cluster {r.number} ({r.act})", name=r["name"], km2=int(m.sum()), mean_f_own=float(F0[m].mean()) if m.any() else np.nan,
                          pct_in_other_core=100 * float((m & core_ab).sum()) / n, pct_in_other_scenario_tier=100 * float((m & (TIERS_AB == 2)).sum()) / n,
                          pct_in_other_opportunity=100 * float((m & (TIERS_AB == 1)).sum()) / n, mean_f_other=float(F0_AB[m].mean()) if m.any() else np.nan))
    CMPC = pd.DataFrame(rows_); CMPC.to_csv(TAB / "comparison_clusters.csv", index=False)
    XT2 = pd.crosstab(pd.Series(TIERS_AB[G.disc], name="Alberta run tier").map(dc.TIER_NAMES), pd.Series(tiers[G.disc], name="Y2Y-wide within Alberta tier").map(dc.TIER_NAMES))
    XT2 = XT2.reindex(index=[dc.TIER_NAMES[k] for k in (3, 2, 1, 0)], columns=[dc.TIER_NAMES[k] for k in (3, 2, 1, 0)], fill_value=0); XT2.to_csv(TAB / "comparison_tiers_crosstab.csv")
    SUMMARY["vs_alberta_run"] = dict(alberta_run_package=str(PKG_AB.relative_to(ROOT)), surfaces=CMP.to_dict(orient="records"), clusters=CMPC.to_dict(orient="records"), tiers_crosstab_km2=XT2.to_dict())
    print(f"\nTHE COMPARISON -- the Alberta run vs the Y2Y-wide analysis within Alberta ({G.n_disc:,} km2 of allocatable land):")
    print(CMP.to_string(index=False, float_format=lambda v: f"{v:,.3f}"))
    print("\ndeck clusters against the other analysis (Act 2 clusters net of their own core):"); print(CMPC.to_string(index=False, float_format=lambda v: f"{v:,.1f}"))
    print("\ntier cross-tab (km2 of allocatable land; rows = the Alberta run, columns = this package):"); print(XT2.to_string())
else:
    print("the Alberta run's package (director_package/) is absent -- run 12 first for the comparison; this package is complete without it")
SUMMARY["forms"] = FORMS; SUMMARY["pool_keys"] = list(POOL)
SUMMARY["ranchland_pool_km2"] = int((ten[G.disc] == 5).sum()); SUMMARY["crown_share_of_unprotected_pct"] = float(100 * np.isin(ten[G.disc], [2, 3]).mean())
(PKG / "summary.json").write_text(json.dumps(SUMMARY, indent=1, default=float))
print(f"\nwrote {PKG.relative_to(ROOT)}/summary.json (VERSION {VERSION}, band {BAND}; the Y2Y-wide analysis within Alberta) -- next: 16_director_outputs_y2y_analysis.ipynb (the presentation set)")


AOI alignment (alignment, not assignment):
                          aoi  km2_unlocked  share_of_unprotected_pct  core_inside_km2  scenario_inside_km2  opportunity_inside_km2  mean_F_inside  mean_F_unprotected  mean_f_s1  mean_f_s2  mean_f_s2c  mean_f_s3  mean_f_s4
Upper Smoky Nature-First zone           436                      0.76               58                   93                     285           0.49                0.17       0.60       0.50        0.39       0.46       0.44
Upper Smoky SRP planning area         10028                     17.54               70                  116                    9808           0.12                0.17       0.12       0.08        0.06       0.19       0.14

tenure by tier (km2; private classes over-counted by the crown-lease share, D-AB8):
         tier   km2  crown (Green Area)  crown (White Area, notated)  private presumed (non-ranch)  private ranchland (OECM track)  unclassified
         core  2141                2038                   